## 머신러닝 기초

> **노트북 사용전 확인할 것**
>
> 셀을 순서대로 실행하지 않으면 변수 상태가 꼬일 수 있음!\
> 위에서 아래 순서대로 실행하면서 확인하자!

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from _style import setup
from _ml import build_dataset, FEATURES, SEED

setup()
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 20)

print("준비 완료")

준비 완료


### 1. 프로그래밍(코딩)과의 차이점

지금까지 우리가 작성해온 프로그래밍은 **규칙을 사람이 정하는 것**이었음!

```python
if 거래량 > 100_000 and 등락률 > 5:
    return "급등주"
```

규칙이 명확할 경우에는 프로그래밍이 빠르고, 정확하고 좋을 수 있음! (설명이 가능)

하지만, **규칙을 사람이 알수 없을 때**는 설명이 어려워짐!
> "오늘 주가가 오를 종목의 조건은 무엇인가?"

거래량, 등락률, 이동평균, ... 수십개 이상의 변수로 조건을 작성할 수 있겠지만,  어려움이 있어
**머신러닝은 데이터에서 그 규칙을 컴퓨터가 찾아내게 하는 방법**이기 때문에 이 문제를 해소해 나갈 수 있음!


In [2]:
df = build_dataset()

# 규칙 : 거래량이 평소보다 2배 이상 많고, 20일 평균보다 3% 이상 높으면 오를 것임!
rule = (df["volume_ratio"] >= 2.0) & (df["ma20_ratio"] >= 1.03)

# df["target_up"] -> 올랐으면 1, 아니면 0
hit = (rule == df["target_up"]).mean()

print(f"규칙에 해당하는 날 : {rule.sum():,}건 / {len(df):,}건")
print(f"규칙의 적중률 : {hit:.4f}")
print(f"전체 상승 비율 : {df['target_up'].mean():.4f}")

규칙에 해당하는 날 : 731건 / 87,480건
규칙의 적중률 : 0.5099
전체 상승 비율 : 0.4895


> **적중률이 생각보다 낮게 나옴!**
>
> 사람이 느낌으로 만든 규칙이기 때문에, 어떤 조건을 어떤 값으로 잡아야하는지
어려움이 있음. 조건을 바꿔가면서 직접 찾을 수도 있지만 만약에 피처(변수)가 7개만 
되어도, 조합이 폭발적으로 많아질 것임!

- 프로그래밍 : 데이터 + 규칙(사람)을 입력해서 결과를 받는다.
- 머신러닝 : 데이터 + 정답을 입력해서 규칙을 만든다.

> 규칙이 명확하면 직접 프로그래밍으로 답을 도출하고,
> **규칙을 알 수 없을 때** 머신러닝을 이용해서 규칙을 만들자!

---
### 분류와 회귀
**무엇을 예측하려고 하는가?**

| | 분류 | 회귀 |
|:--:|:--:|:--:|
| 예측 가능한 것 | 범주 | 수치 |
| 질문 | 오를까? 내릴까? | 얼마나 오를까? |
| 출력 | up / down | 2.7% |
| 평가지표 | 정확도, 정밀도, 재현율 | MAE, RMSE, ... |

**동일한 데이터로 두 문제를 모두 도출할 수 있음!**

In [3]:
# target_ret : 수익률 - 수치 => 회귀
# target_up : 올랐는지, 오르지 않았는지 (1/0) - 범주 => 분류

sample = df[["code","date","target_ret","target_up"]].head(8)
print(sample.to_string(index=False))

print("같은 행이지만 정답을 두 가지로 확인할 수 있음! (회귀/분류)")

 code       date  target_ret  target_up
G0001 2023-10-24    0.015574          1
G0001 2023-10-25   -0.008639          0
G0001 2023-10-26   -0.002960          0
G0001 2023-10-27    0.002676          1
G0001 2023-10-30    0.003128          1
G0001 2023-10-31    0.046732          1
G0001 2023-11-01   -0.001827          0
G0001 2023-11-02    0.011819          1
같은 행이지만 정답을 두 가지로 확인할 수 있음! (회귀/분류)


> **평가 지표는 섞어서 사용할 수 없음**
>
> 분류 모델에 MAE, RMSE, ... 을 사용하거나 \
> 회귀 모델에 정확도를 사용할 수 있을까요..? 사용할 수 없음! \
> "정확도 92%" 라는 판단은 **분류에서만** 의미가 있으므로, 회귀에서는  "정확하게
맞췄다"는 개념 자체가 성립되지 않는다. (회귀에서는 오차만 확인) 

### 기준선 -> 사람이 정함!

연속된 값을 범주로 변경하려면 어디서 자를것인가?를 정해야 함 -> **정답의 기준이 될것임**

In [4]:
# 기준선을 바꿔 가면서 "상승"의 비율이 어떻게 변하는지 확인

base_line = [0.0, 0.01, 0.02, 0.03, 0.05]

print(f"{'기준선':>10}{'양성비율':>12}{'양성건수':>12}")
print("-" * 60)

for th in base_line:
    positive = (df["target_ret"] > th)
    print(f"{th:>9.0%}{positive.mean():>12.2%}{positive.sum():>12,}")

       기준선        양성비율        양성건수
------------------------------------------------------------
       0%      48.95%      42,825
       1%      29.46%      25,773
       2%      15.30%      13,387
       3%       6.74%       5,894
       5%       1.38%       1,210


> **기준선이 결과를 바꾼다**
>
> 0%을 기준으로 하면 상승, 하락이 거의 반반이다. \
> 5%를 기준으로 하면 상승이 1.4% 밖에 안된다. \
> 이런 불균형 데이터에서 "전부 하락"이라고 하면 정확도가 98%가 넘을 것임!

### 학습, 검증 데이터 분리

시험 문제를 미리 알려주고 시험을 보면 당연히 점수가 높게 나올 것임.\
모델도 동일하다. **학습에 사용한 데이터로 평가하면, 성능이 과대평가**될 것임!

|데이터|용도|비율|
|:--:|:--:|:--:|
|학습|모델이 규칙을 학습(배움)|60~70%|
|검증|하이퍼 파라미터 조정|15~20%|
|테스트|최종 성능 측정|15~20%|

scikit-learn 모듈 설치
```bash
pip install scikit-learn
```

In [5]:
from sklearn.model_selection import train_test_split

small = df.sample(5000, random_state=SEED)

X = small[FEATURES]     # 입력 데이터(피처, 독립변수)
y = small["target_up"]  # 정답 데이터(레이블, 종속변수)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,               # 입력, 정답
    test_size=0.2,      # 학습 8 : 테스트 2
    random_state=SEED,  # 재현성을 위해 SEED를 정해줌
    stratify=y,         # 타깃 레이블(y) 클래스 비율을 그대로 유지한 채 학습 데이터와 테스트 데이터를 분리
)

print(f"X shape : {X.shape} (행수,피처수)")
print(f"y shape : {y.shape} (1차원)")
print()
print(f"학습 : {len(X_train):,}행 / 상승 비율 : {y_train.mean():.4f}")
print(f"테스트 : {len(X_test):,}행 / 상승 비율 : {y_test.mean():.4f}")

X shape : (5000, 7) (행수,피처수)
y shape : (5000,) (1차원)

학습 : 4,000행 / 상승 비율 : 0.4995
테스트 : 1,000행 / 상승 비율 : 0.4990


---

### 시계열 데이터는 랜덤 분할하면 안됨!

시세 데이터는 날짜가 있는 시계열 데이터임. 그냥 이전과 동일하게 train_test_split 함수를 사용하면 문제가 될
수 있음!

In [6]:
# 랜덤 분할
rnd_train, rnd_test = train_test_split(df, test_size=0.2, random_state=SEED)

# 시점 분할 
# -> 직접 날짜를 기준으로 앞 뒤로 자른다.
cutoff = df["date"].quantile(0.8)   # 상위 80% 지점의 날짜

seq_train = df[df["date"] <= cutoff]
seq_test = df[df["date"] > cutoff]

print("===랜덤 분할 ===")
print(f"{rnd_train.date.min().date()} ~ {rnd_train.date.max().date()}")
print(f"{rnd_test.date.min().date()} ~ {rnd_test.date.max().date()}")
print()
print("===시점 분할 ===")
print(f"{seq_train.date.min().date()} ~ {seq_train.date.max().date()}")
print(f"{seq_test.date.min().date()} ~ {seq_test.date.max().date()}")

===랜덤 분할 ===
2023-10-24 ~ 2026-09-07
2023-10-24 ~ 2026-09-07

===시점 분할 ===
2023-10-24 ~ 2026-01-16
2026-01-19 ~ 2026-09-07


---

### scikit-learn 공통 인터페이스
**모든 모델이 같은 방식으로 동작하는 것**이 scikit-learn의 큰 장점임!

```python
model = 모델생성()

model.fit(X_train, y_train)          # 학습
pred = model.predict(X_test)         # 예측
score = model.score(X_test, y_test)  # 평가
```

In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# 시점 분할한 데이터 사용
train_s = seq_train.sample(8000, random_state=SEED)
test_s = seq_test.sample(2000, random_state=SEED)

Xtr, ytr = train_s[FEATURES], train_s["target_up"]
Xte, yte = test_s[FEATURES], test_s["target_up"]

# 모델 생성
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=SEED),
    "DecisionTree": DecisionTreeClassifier(max_depth=5, random_state=SEED),
    "RandomForest": RandomForestClassifier(max_depth=5, random_state=SEED,
                                           n_estimators=50, n_jobs=-1),
}

for name, model in models.items():
    model.fit(Xtr, ytr)              # 학습 (입력, 정답) => 학습용
    pred = model.predict(Xte)        # 예측 (입력) => 검증용/테스트용
    score = model.score(Xte, yte)    # 평가 (입력, 정답) => 검증용/테스트용

    print(f"{name:<20} score {score:.4f}")

LogisticRegression   score 0.5025
DecisionTree         score 0.4890
RandomForest         score 0.4945


|메소드|역할|
|:--:|:--:|
|fit(X,y)|학습|
|predict(X)|예측|
|predict_proba(X)|확률(분류)|
|score(X,y)|기본평가지표|

> score() 는 모델마다 의미가 달라짐!\
> 분류 모델에서는 정확도, 회귀 모델이면 R²(결정계수)를 반환

In [8]:
# predict 와 predict_proba 비교

rf = models["RandomForest"]

pred = rf.predict(Xte)      # 0 또는 1
proba = rf.predict_proba(Xte)   # 각 클래스에 속할 확률  

print(f"predict      shape{pred.shape}     앞 5개 {pred[:5]}")
print(f"predict_proba     shape{proba.shape}")
print()
print(f"predict_proba 의 앞 5행")
print(pd.DataFrame(proba[:5], columns=["하락일 확률", "상승일 확률"]).round(4))


predict      shape(2000,)     앞 5개 [1 0 1 0 0]
predict_proba     shape(2000, 2)

predict_proba 의 앞 5행
   하락일 확률  상승일 확률
0  0.4860  0.5140
1  0.5471  0.4529
2  0.4901  0.5099
3  0.5353  0.4647
4  0.5477  0.4523


---

- 머신러닝 : 규칙을 알 수 없을 때 데이터에서 규칙을 찾아내는 방법
- 분류 vs 회귀 : 범주? 수치?  -> 평가 지표가 다르게 나옴!
- 데이터 분할 : 학습용 데이터와 테스트용 데이터로 나누어 사용, 테스트 데이터는 마지막에 한 번만 사용.
  - 시계열 데이터 : 랜덤 금지!. 시점을 기준으로 구분해야 함!
- 과적합 : 학습과 테스트 결과가 차이가 심할 때!  